# TOAD × SAM3 — Colab Training

Finetunes SAM3 on TOAD histology segmentation data.

**Before running:**
1. Runtime → Change runtime type → GPU (T4 or better)
2. Upload `seko_storage/` to your Google Drive (keep the same folder structure)

## 0. Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import subprocess, sys, os
import numpy as np

if np.__version__ >= "2":
    print(f"Colab has numpy {np.__version__} — downgrading to <2 (required by sam3)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy<2"], check=True)
    print("Restarting runtime — run all cells again after restart.")
    os.kill(os.getpid(), 9)  # kills kernel; Colab auto-restarts it
else:
    print(f"numpy {np.__version__} OK — no restart needed.")

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess

REPO_URL    = "https://github.com/DreamTeamSE/TOAD.git"
REPO_BRANCH = "feature/sam-3"
TOAD_DIR    = "/content/TOAD"

if not os.path.isdir(TOAD_DIR):
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--depth", "1",
                    REPO_URL, TOAD_DIR], check=True)
    print(f"Cloned {REPO_BRANCH} → {TOAD_DIR}")
else:
    subprocess.run(["git", "-C", TOAD_DIR, "pull"], check=True)
    print("Repo already present — pulled latest.")

## 2. Configuration

Edit these variables to match your setup.

In [ ]:
import os

# ── Data locations (Google Drive) ───────────────────────────────────────────
STORAGE_DIR = "/content/drive/MyDrive/seko_storage"  # seko_storage root on Drive

TRAIN_CSV  = f"{STORAGE_DIR}/data/rat_original/train.csv"
OUTPUT_DIR = f"{STORAGE_DIR}/outputs"

# ── Checkpoint (persisted on Drive) ─────────────────────────────────────────
CKPT_DIR  = "/content/drive/MyDrive/checkpoints"
CKPT_PATH = f"{CKPT_DIR}/sam3.pt"

# ── Training hyperparams ────────────────────────────────────────────────────
EPOCHS   = 20
LR       = 1e-4
VAL_FRAC = 0.15
SEED     = 42

print("Config:")
print(f"  TOAD dir   : {TOAD_DIR}")
print(f"  train CSV  : {TRAIN_CSV}")
print(f"  output dir : {OUTPUT_DIR}")
print(f"  checkpoint : {CKPT_PATH}")
print(f"  epochs     : {EPOCHS}  lr={LR}  val_frac={VAL_FRAC}")

assert os.path.isfile(TRAIN_CSV), f"train.csv not found: {TRAIN_CSV}"
print("\ntrain.csv found ✓")

## 3. Install Dependencies

In [ ]:
# Colab already has: torch, torchvision, numpy, pandas, scipy, scikit-learn,
#                    opencv-python, pillow, huggingface_hub
# We only need to add the SAM3-specific extras.
!pip install -q "numpy<2" \
    timm==1.0.25 \
    einops==0.8.2 \
    iopath==0.1.10 \
    hydra-core==1.3.2 \
    omegaconf==2.3.0 \
    ftfy==6.1.1 \
    pycocotools \
    decord \
    torchmetrics==1.9.0 \
    supervision==0.27.0.post1

print("Dependencies installed.")

## 4. Install SAM3 Package

Installs from the official Meta GitHub repo (`facebookresearch/sam3`).

In [ ]:
!pip install -q git+https://github.com/facebookresearch/sam3.git
print("sam3 installed.")

### 4a. Patch `sam3/model/edt.py`

The upstream package has a SyntaxError from duplicate/dangling triton imports.
This cell rewrites the problematic section with a clean try/except guard.

In [ ]:
import importlib, pathlib, re

import sam3
edt_path = pathlib.Path(sam3.__file__).parent / "model" / "edt.py"
print(f"Patching: {edt_path}")

edt_src = edt_path.read_text()

PATCH_MARKER = "# TOAD-PATCH: triton guard applied"
if PATCH_MARKER in edt_src:
    print("Already patched — skipping.")
else:
    TRITON_GUARD = f"""\
{PATCH_MARKER}
HAS_TRITON = False
try:
    import triton
    import triton.language as tl
    HAS_TRITON = True
except (ImportError, Exception):
    pass
"""
    # Strip the broken triton import lines
    cleaned = re.sub(r'^\s*(?:try:\s*)?import triton.*?(?:\.language as tl)?.*$',
                     '', edt_src, flags=re.MULTILINE)
    cleaned = re.sub(r'^\s*import triton\.language as tl\s*$',
                     '', cleaned, flags=re.MULTILINE)

    # Insert the clean guard after the opening comments/docstring
    lines = cleaned.splitlines(keepends=True)
    insert_at = 0
    for i, line in enumerate(lines):
        if line.strip().startswith('#') or line.strip() == '' \
                or line.strip().startswith('"""') or line.strip().startswith("'''"):
            insert_at = i + 1
        else:
            break
    lines.insert(insert_at, TRITON_GUARD + '\n')
    edt_path.write_text(''.join(lines))
    print("edt.py patched.")

# Verify
try:
    import sam3.model.edt
    importlib.reload(sam3.model.edt)
    print("sam3.model.edt imports OK.")
except SyntaxError as e:
    print(f"SyntaxError still present — check edt.py manually: {e}")

## 5. Download SAM3 Checkpoint

In [ ]:
os.makedirs(CKPT_DIR, exist_ok=True)

if os.path.isfile(CKPT_PATH):
    size_mb = os.path.getsize(CKPT_PATH) / 1e6
    print(f"Checkpoint already exists ({size_mb:.0f} MB) — skipping download.")
else:
    print("Downloading sam3.pt from HuggingFace ...")
    from huggingface_hub import hf_hub_download
    hf_hub_download(
        repo_id="facebook/sam3",
        filename="sam3.pt",
        local_dir=CKPT_DIR,
    )
    print(f"Checkpoint saved to {CKPT_PATH}")

## 6. Quick Environment Verification

In [ ]:
import torch, cv2, numpy as np, pandas as pd
from sam3 import build_sam3_image_model
from sam3.model.data_misc import BatchedFindTarget, FindStage
from sam3.model.geometry_encoders import Prompt
from sam3.train.loss.loss_fns import dice_loss
from scipy.optimize import linear_sum_assignment

print("All imports OK.")
print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")

df = pd.read_csv(TRAIN_CSV)
print(f"train.csv: {len(df)} rows, columns={list(df.columns)}")

## 7. Run Training

## 7a. Regenerate train.csv with Colab Paths

In [ ]:
import subprocess, sys

# Regenerate train.csv with absolute Colab paths
data_root = f"{STORAGE_DIR}/data/rat_original"
subprocess.run([
    sys.executable,
    os.path.join(TOAD_DIR, "src", "utils", "build_train_csv.py"),
    "--data-root", data_root,
    "--out", TRAIN_CSV,
], check=True)
print(f"train.csv regenerated at {TRAIN_CSV}")

In [ ]:
import subprocess, sys, shlex

train_script = os.path.join(TOAD_DIR, "src", "train_sam3.py")

cmd = [
    sys.executable, train_script,
    "--csv",        TRAIN_CSV,
    "--output-dir", OUTPUT_DIR,
    "--ckpt",       CKPT_PATH,
    "--device",     "cuda",
    "--epochs",     str(EPOCHS),
    "--lr",         str(LR),
    "--val-frac",   str(VAL_FRAC),
    "--seed",       str(SEED),
]

print("Running:", shlex.join(cmd), "\n")

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()

if proc.returncode == 0:
    print("\nTraining complete.")
else:
    print(f"\nTraining exited with code {proc.returncode}.")

## 8. Training Metrics

In [ ]:
import json, os
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

metrics_path = os.path.join(OUTPUT_DIR, "metrics.json")
assert os.path.isfile(metrics_path), f"metrics.json not found at {metrics_path}"

with open(metrics_path) as f:
    m = json.load(f)

epochs     = m["epoch"]
train_loss = m["train_loss"]
val_loss   = m["val_loss"]
train_dice = m["train_dice"]
val_dice   = m["val_dice"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("SAM3 Fine-tuning on TOAD Histology", fontsize=14, fontweight="bold")

# ── Loss ──────────────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(epochs, train_loss, marker="o", markersize=3, label="Train loss")
ax.plot(epochs, val_loss,   marker="o", markersize=3, label="Val loss")
ax.set_title("Dice + BCE Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.grid(alpha=0.3)

# ── DICE ──────────────────────────────────────────────────────────────────
ax = axes[1]
ax.plot(epochs, train_dice, marker="o", markersize=3, label="Train DICE")
ax.plot(epochs, val_dice,   marker="o", markersize=3, label="Val DICE")
ax.set_title("DICE Score (higher is better)")
ax.set_xlabel("Epoch")
ax.set_ylabel("DICE")
ax.set_ylim(0, 1)
ax.legend()
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

best_epoch = val_dice.index(max(val_dice)) + 1
print(f"
Epoch 1  → val DICE = {val_dice[0]:.4f}")
print(f"Epoch {best_epoch:2d} → val DICE = {max(val_dice):.4f}  (best)")
print(f"Improvement: {max(val_dice) - val_dice[0]:+.4f}")


## 9. Inspect Saved Checkpoints

In [ ]:
import glob

ckpt_glob = os.path.join(OUTPUT_DIR, "sam3_checkpoints", "*.pt")
checkpoints = sorted(glob.glob(ckpt_glob))

if not checkpoints:
    print(f"No checkpoints found at {ckpt_glob}")
else:
    print(f"Found {len(checkpoints)} checkpoint(s):")
    for p in checkpoints:
        ckpt = torch.load(p, map_location="cpu", weights_only=True)
        epoch    = ckpt.get('epoch', '?')
        val_loss = ckpt.get('val_loss', float('nan'))
        size_mb  = os.path.getsize(p) / 1e6
        print(f"  {os.path.basename(p):40s}  epoch={epoch:>3}  val_loss={val_loss:.4f}  {size_mb:.0f} MB")